In [4]:
pip install --upgrade diffusers

In [19]:
#!/usr/bin/env python3
"""
pretrained_sd_clip_gate.py — Pretrained Stable Diffusion generator with CLIP gating
Colab-friendly, no fine-tuning.

- Starts from white-noise latents
- Repeats until CLIP score passes threshold
- Saves output.png (and best.png optionally)
"""

import os, json, time
import torch
from PIL import Image
import numpy as np

from diffusers import DiffusionPipeline
from transformers import CLIPModel, CLIPProcessor

# -----------------------------
# Config (adjust if needed)
# -----------------------------

PROMPT = "Elephant"
NEGATIVE_PROMPT = "clear, mid quality, artifacts, accurate"
HEIGHT, WIDTH = 256, 256        # low-VRAM
STEPS = 45                      # faster, lighter
GUIDANCE = 10.5
THRESHOLD = 0.35                # acceptance CLIP cosine
MAX_ITERS = 100
SEED = 42
OUT_PATH = "output.png"
SAVE_BEST = True
SAVE_LOG = True

# -----------------------------
# Utilities
# -----------------------------

def resolve_device_dtype():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required.")
    return torch.device("cuda"), torch.float16  # fp16 to save VRAM

def gen_white_noise_png(path="white-noise.png", width=WIDTH, height=HEIGHT):
    arr = (np.random.rand(height, width, 3) * 255).astype("uint8")
    Image.fromarray(arr).save(path)
    return path

def load_sd_pipeline(dtype, device):
    pipe = DiffusionPipeline.from_pretrained(
      "stable-diffusion-v1-5/stable-diffusion-v1-5",
      torch_dtype=torch.float16,
      safety_checker=None,
    )
    pipe.to(device)

    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        print("[warn] xformers not available; continuing without it.")
    pipe.set_progress_bar_config(disable=True)
    return pipe

def load_clip(device):
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    return model, processor

def make_white_noise_latents(height, width, dtype, device, seed=None):
    if height % 8 != 0 or width % 8 != 0:
        raise ValueError("Height and width must be divisible by 8 for SD.")
    g = torch.Generator(device=device).manual_seed(seed if seed is not None else torch.seed())
    latents = torch.randn((1, 4, height // 8, width // 8), generator=g, device=device, dtype=dtype)
    return latents, g

def to_pil(image_tensor: torch.Tensor):
    imgs = image_tensor.clamp(0,1).detach().cpu()
    pil_list = []
    for i in range(imgs.size(0)):
        arr = (imgs[i].permute(1,2,0).numpy() * 255).astype("uint8")
        pil_list.append(Image.fromarray(arr))
    return pil_list

def clip_cosine_score(model, processor, image: Image.Image, prompt: str, device: torch.device) -> float:
    inputs = processor(text=[prompt], images=[image], return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs)
        img_emb = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
        txt_emb = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
        sim = (img_emb * txt_emb).sum(dim=-1)
    return float(sim.item())

def save_image(img: Image.Image, path: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    img.save(path)

# -----------------------------
# Main gating loop
# -----------------------------

def run():
    device, dtype = resolve_device_dtype()
    pipe = load_sd_pipeline(dtype, device)
    clip_model, clip_proc = load_clip(device)

    # start with a local seed
    seed = 42
    rng = torch.Generator(device=device).manual_seed(seed)

    best_score = -1.0
    best_img = None
    scores = []

    for i in range(1, MAX_ITERS + 1):
        # generate latents with current seed
        latents = torch.randn((1, 4, HEIGHT // 8, WIDTH // 8),
                              generator=rng, device=device, dtype=dtype)

        out = pipe(prompt=PROMPT,
                   negative_prompt=NEGATIVE_PROMPT,
                   num_inference_steps=STEPS,
                   guidance_scale=GUIDANCE,
                   latents=latents)
        img = out.images[0]

        sim = clip_cosine_score(clip_model, clip_proc, img, PROMPT, device)
        scores.append({"iter": i, "clip_score": sim})

        if sim > best_score:
            best_score = sim
            best_img = img.copy()

        print(f"[iter {i}] score={sim:.4f}")

        if sim >= THRESHOLD:
            save_image(img, OUT_PATH)
            return

        # advance seed for next iteration
        seed += 1
        rng = torch.Generator(device=device).manual_seed(seed)

    # fallback
    if best_img is not None:
        save_image(best_img, OUT_PATH)

# -----------------------------
# Entry
# -----------------------------

if __name__ == "__main__":
    run()


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[warn] xformers not available; continuing without it.
[iter 1] score=0.2237
[iter 2] score=0.2159
[iter 3] score=0.2005
[iter 4] score=0.2298
[iter 5] score=0.2540
[iter 6] score=0.2121
[iter 7] score=0.3002
[iter 8] score=0.3111
[iter 9] score=0.2471
[iter 10] score=0.2456
[iter 11] score=0.1770
[iter 12] score=0.2278
[iter 13] score=0.3002
[iter 14] score=0.2273
[iter 15] score=0.2048
[iter 16] score=0.2365
[iter 17] score=0.2926
[iter 18] score=0.2346
[iter 19] score=0.3035
[iter 20] score=0.2544
[iter 21] score=0.2282
[iter 22] score=0.2782
[iter 23] score=0.3075
[iter 24] score=0.2404
[iter 25] score=0.2580
[iter 26] score=0.2876
[iter 27] score=0.3066
[iter 28] score=0.2856
[iter 29] score=0.3094
[iter 30] score=0.2791
[iter 31] score=0.2651
[iter 32] score=0.2168
[iter 33] score=0.3047
[iter 34] score=0.2272
[iter 35] score=0.2077
[iter 36] score=0.2752
[iter 37] score=0.3121
[iter 38] score=0.2881
[iter 39] score=0.2964
[iter 40] score=0.2614
[iter 41] score=0.3137
[iter 42] sc